In [1]:
# MeCab, MeCab用の辞書, tf-idf用機能, ワードクラウド, ワードクラウドグラフを表示するための機能をインストール
!pip install mecab-python3 unidic-lite scikit-learn wordcloud matplotlib
# フォントをインストール
!apt-get install fonts-noto-cjk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 20.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 13.2 MB/s eta 0:00:0000:01
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=5dabd149f95ca9c7dab1b9be58d50d1a7da825e3cd8aee529d955b9da3a72b0b
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  fonts-noto-cjk-extra
The following NEW packages will be installed:
  fonts-noto-cjk
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 61.2 MB of archives.
After this operation, 93.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-cjk all 1:20220127+repack1-1 [61.2 MB]
Fetched

In [2]:
# フォントリストを出力
!fc-list :lang=ja

/usr/share/fonts/opentype/noto/NotoSerifCJK-Bold.ttc: Noto Serif CJK SC:style=Bold
/usr/share/fonts/opentype/noto/NotoSerifCJK-Bold.ttc: Noto Serif CJK TC:style=Bold
/usr/share/fonts/opentype/noto/NotoSerifCJK-Bold.ttc: Noto Serif CJK JP:style=Bold
/usr/share/fonts/opentype/noto/NotoSerifCJK-Bold.ttc: Noto Serif CJK HK:style=Bold
/usr/share/fonts/opentype/noto/NotoSerifCJK-Bold.ttc: Noto Serif CJK KR:style=Bold
/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc: Noto Sans CJK JP:style=Regular
/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc: Noto Sans CJK HK:style=Regular
/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc: Noto Sans CJK KR:style=Regular
/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc: Noto Sans CJK SC:style=Regular
/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc: Noto Sans CJK TC:style=Regular
/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc: Noto Serif CJK SC:style=Regular
/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc: Noto

In [ ]:
# ファイルをアップロード
from google.colab import files
files.upload()

In [ ]:
# --- 複数文書用：TF-IDFワードクラウド（Google Colab対応） ---
# 目的：複数のテキストそれぞれについて TF-IDF 上位語をワードクラウドで可視化する（5作品を1枚ずつ表示）

# ✅ 必要ライブラリの読み込み
import MeCab
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer

# ✅ MeCab（分かち書き）
mecab = MeCab.Tagger("-Owakati")

# ✅ ここをあなたのファイル名に変更（Colabにアップロード済みの .txt を指定）
file_names = [
    "akutagawa_rashomon.txt",
    "atsushi_sangetsuki.txt",
    "kajii_remon.txt",
    "ohgai_maihime.txt",
    "shiga_kinosakinite.txt",
]

# ✅ テキスト読込
texts = []
for fn in file_names:
    with open(fn, "r", encoding="utf-8") as f:
        texts.append(f.read())

# ✅ 形態素解析（分かち書き）
wakati_documents = [mecab.parse(doc).strip() for doc in texts]

# ✅ TF-IDF計算（複数文書）
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(wakati_documents)  # shape: (n_docs, vocab_size)
feature_names = vectorizer.get_feature_names_out()

# ✅ 日本語フォント（Colab標準のNoto）
FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

def make_wordcloud_from_scores(words, scores, topn=200):
    """単語とスコア（同じ長さの配列）からワードクラウドを作る"""
    scores = np.asarray(scores).flatten()

    # スコアが0の語は除外、上位topn語に絞る（表示が安定）
    nz = scores > 0
    words_nz = np.asarray(words)[nz]
    scores_nz = scores[nz]

    if len(scores_nz) == 0:
        raise ValueError("スコア>0の単語がありません。分かち書きや入力ファイルを確認してください。")

    idx = np.argsort(scores_nz)[::-1][:topn]
    freq = {words_nz[i]: float(scores_nz[i]) for i in idx}

    wc = WordCloud(
        font_path=FONT_PATH,
        background_color="white",
        width=900,
        height=650
    ).generate_from_frequencies(freq)

    return wc, freq

# ✅ 文書ごとにワードクラウドを作成
wordclouds = []
top_terms = []

n_docs = tfidf_matrix.shape[0]

for i in range(n_docs):
    scores_i = tfidf_matrix[i].toarray().flatten()
    wc, freq = make_wordcloud_from_scores(feature_names, scores_i, topn=200)
    wordclouds.append(wc)

    # 上位20語も一緒に確認できるように保存
    top20 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:20]
    top_terms.append(top20)

# ✅ 表示（1作品ずつ別々に表示）
for i, wc in enumerate(wordclouds):
    plt.figure(figsize=(10, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(file_names[i], fontsize=14)
    plt.tight_layout()
    plt.show()

# ✅ 上位語（20語）をテキストでも表示（授業・確認用）
for i, top20 in enumerate(top_terms):
    print(f"\n--- 文書{i+1} 上位20語（TF-IDF）: {file_names[i]} ---")
    for w, s in top20:
        print(f"{w}\t{s:.4f}")